# Homework 5 - Bayesian Classification Model

Make sure you have downloaded:
- heart_processed_log.csv

This homework will ask you to implement naive bayes using a custom likelihood and then comparing it against a custom build LDA and QDA implimentation. 

The execution of GNB is slightly different from lecture and section. 
- It is more streamlined to take adavantage of vector multiplications and numpy functions, which has its own benefits if we want to scale up our naive bayes prediction to higher dimensions. 
- However, you may need to familiarize yourself with the "dictionary" data structure.

Before attempting this homework, make sure you understand the broad strokes of naive Bayes. This will make your coding and debugging much smoother.

## AI Assistance
We understand that we cannot stop you from using AI to help you with your homework. We also recognize that this is a skill that will be useful in the real world. With that in mind, we allow the assitence of AI on your homework. 

#### <span style="color:red">HOWEVER, you must include the prompt you used to get the answer in your submission. Additionally, you should give a brief explanation of what worked and what didn't with regard to the AI.</span>

There will be an additional cell after each question so that you can include your prompt and explanation.

Some homework assignments may also include a problem in which you are explicity asked to use AI. You must still include your prompt and explanation in the cell provided.

##### <span style="color:blue">Note: Exams are going to be on paper and proctored (i.e. without the use of AI). If you rely too heavily on AI assistance to complete your homework you may see determental effects on your exam performance.</span>

# 0 Data
Load `heart_processed.csv` from the [Heart Failure Clinical Records Dataset](https://archive.ics.uci.edu/ml/datasets/Heart%2Bfailure%2Bclinical%2Brecords)  It contains various predictors (which are in log-scale) for predicting the event of death `DEATH_EVENT`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from scipy.stats import multivariate_normal

Before submitting your homework, remember to set:
- random_state = 0

In [2]:
random_state = 0

In [92]:
dataset = pd.read_csv("heart_processed_log.csv", index_col=0)
display(dataset)

X = dataset.drop("DEATH_EVENT", axis=1).values
y = dataset["DEATH_EVENT"].values

# split the data into training and testing sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=600000)

# print the shapes of the training and testing sets
print('train shapes:')
print('\t X_train ->', X_train.shape)
print('\t y_train ->', y_train.shape)

print('test shapes:')
print('\t X_test ->', X_test.shape)
print('\t y_test ->', y_test.shape)

,age,creatinine_phosphokinase,ejection_fraction,platelets,serum_creatinine,serum_sodium,DEATH_EVENT
0,4.317488,6.366470,2.995732,12.487485,0.641854,4.867534,1
1,4.007333,8.969669,3.637586,12.481270,0.095310,4.912655,1
2,4.174387,4.983607,2.995732,11.995352,0.262364,4.859812,1
3,3.912023,4.709530,2.995732,12.254863,0.641854,4.919981,1
4,4.174387,5.075174,2.995732,12.697715,0.993252,4.753590,1
...,...,...,...,...,...,...,...
294,4.127134,4.110874,3.637586,11.951180,0.095310,4.962845,0
295,4.007333,7.506592,3.637586,12.506177,0.182322,4.934474,0
296,3.806662,7.630461,4.094345,13.517105,-0.223144,4.927254,0
297,3.806662,7.788626,3.637586,11.849398,0.336472,4.941642,0


train shapes:
	 X_train -> (209, 6)
	 y_train -> (209,)
test shapes:
	 X_test -> (90, 6)
	 y_test -> (90,)


Recall: naive Bayes is choosing the class $k$, $C_k$, that maximizes the posterior
$$
P(C_k \lvert\,{x}) = \frac{\pi(C_k)\,{\cal{}L}_{\!{x}}(C_k)}{Z}.
$$
Hence, we maximize the numerator + assume that all $d$ features $x_i$ are independent ("naive-ness"). So we want to find the $k$ that satisfies
$$
\max_k \, \pi(C_k)\,{\cal{}L}_{\!{x}}(C_k) \quad = \quad \max_k \, \left( \pi(C_k)\,\prod_{i=1}^d p(x_i \lvert C_k) \right).
$$

# 1 Custom Naive Bayes Classifier with KDE
You will create a naive Bayes classifier:
- using the training data
- with KDE to approximate the likelihood
- with bernoulli as the prior

**Use only the training data ```X_train, y_train``` to fit the naive Bayes classifier.**

## 1.1 Prior
**Task:**

[2 pts] Compute ```prior```, a two element array. 
    - prior[0] is the probability of death event 0, $\pi(C_0)$
    - prior[1] is the probability of death event 1, $\pi(C_1)$ 
    - You should construct the prior probabilities based on frequency of death events from the training data. 
    - Tip: Use np.unique() with return_counts.

In [78]:
prior = np.unique(y, return_counts = True)[1] / np.size(y)      # TODO Compute prior

print('The prior probabilities are:', prior)

The prior probabilities are: [0.67892977 0.32107023]


### 1.1 AI Analysis (if used)

#### Prompt(s):

#### Explanation / Changes made:

## 1.2 Likelihood (KDE)

**Task:**

1. [2 pts] Define dictionaries `kde0` and `kde1` which fulfill the following:
    - kde0[i] corresponds to the kde object (created by calling `scipy.stats.gaussian_kde`) for feature i when death event is 0. kde1[i] defined likewise.
    - Make sure you index the correct rows of `X_train` when defining kdes.
    - Use bandwidth method 'scott'. (For fun, you can try 'silverman' and see what difference in result you get.)
    - As with all arrays you throw into sklearn or scipy, you may need to take transposes.

In [79]:
kde0 = {} 
kde1 = {} 

train_features0 = X_train[y_train==0,:]
train_features1 = X_train[y_train==1,:]

for i in range(X_train.shape[1]):
    kde0[i] = gaussian_kde(train_features0[:, i], bw_method='scott') # TODO construct KDE object for class 0
    kde1[i] = gaussian_kde(train_features1[:, i], bw_method='scott') # TODO construct KDE object for class 1

display(kde1) # Use this to check what you made. swap kde0 for kde1 if you want

{0: <scipy.stats._kde.gaussian_kde at 0x7453a5b52ad0>,
 1: <scipy.stats._kde.gaussian_kde at 0x7453a5b52140>,
 2: <scipy.stats._kde.gaussian_kde at 0x7453a5e0da20>,
 3: <scipy.stats._kde.gaussian_kde at 0x7453a39143a0>,
 4: <scipy.stats._kde.gaussian_kde at 0x7453a39148e0>,
 5: <scipy.stats._kde.gaussian_kde at 0x7453a3915390>}

2. [2 pt] Complete the code for ```compute_likelihood``` function.
    - The objects kde0[i] and kde1[i] have a method .pdf(), which you will use when computing the likelihood.
        - Read the documentation to understand how it works.
    - `likelihood0[j]` is the likelihood of seeing $j$ th data ${x_j} = \left({x_j}_1, \dots, {x_j}_d\right)$ for death event 0, i.e., ${L}_{{x_j}}(C_0) = \prod_{i=1}^d p({x_j}_i | C_0)$
    - `likelihood1[j]` defined likewise.
    - You can loop over the kde objects kde[i] to populate the likelihood arrays.
    - Be careful with the shape of arrays. Print shapes as necessary when debugging.

(Your solution shouldn't be very complicated. A working solutions needs only about 5-10 lines of code.)

In [80]:
def compute_likelihood(x, kde0, kde1):
    # input:    x, a (# data) by (# features) array of test data
    #           kde0 and kde1, dictionaries that will be used to compute the likelihood
    # output:   likelihood, a (# data) by (# classes) array. 
    #           likelihood[j,k] is the likelihood of data j given class k
    
    # likelihood0[j] is the likelihood of data j given class 0. Analogously for likelihood1
    likelihood0 = np.ones(x.shape[0])    
    likelihood1 = np.ones(x.shape[0])    

    for i in range(x.shape[1]):
        likelihood0 *= kde0[i].pdf(x[:,i]) # TODO compute likelihood for class 0
        likelihood1 *= kde1[i].pdf(x[:,i]) # TODO compute likelihood for class 1

    likelihood = np.vstack((likelihood0, likelihood1)).T
    
    return likelihood


### 1.2 AI Analysis (if used)

#### Prompt(s):

#### Explanation / Changes made:

## 1.3 Posterior
**Task:**

[2 pts] Complete the code for ```compute_posterior``` function. 
- It should include calling the function ```compute_likelihood```.

In [81]:
def compute_posterior(x, prior, kde0, kde1):
    # input:    x, a (# data) by (# features) array of test data
    #           prior, a 1 by 2 array
    #           kde0 and kde1, kde dictionaries that will be used to compute the likelihood
    # output:   posterior, a (# data) by (# classes) array

    likelihood = compute_likelihood(x, kde0, kde1) # TODO obtain likelihood
    posterior = (likelihood * prior) / np.sum(likelihood * prior, axis=1, keepdims=True) # TODO compute posterior
    # the normalization constant is axis=1 because we want to sum over the classes
    
    return posterior

### 1.3 AI Analysis (if used)

#### Prompt(s):

#### Explanation / Changes made:

## 1.4 Combine prior, likelihood, posterior
Now, we are ready to piece all the code we prepared above.

**Task:**
1. [2 pts] Complete the code for ```naive_bayes_predict```.
    - Your code should include calling the ```compute_posterior``` function.
    - Computing y_pred should be a simple one line of code. You may consider using numpy functions that find the index of the largest entry on every row.
2. [1 pt] Complete the code for ```print_success_rates```.

In [82]:
def naive_bayes_predict(x, prior, kde0, kde1):
    # input:    x, a (# data) by (# features) array
    #           prior, a 1 by 2 array
    #           kde0 and kde1, kde dictionaries that will be used to compute the likelihood
    # output:   y_pred, an array of length (# data)

    posterior = compute_posterior(x, prior, kde0, kde1) # TODO compute posterior
    y_pred = np.argmax(posterior, axis=1) # TODO make prediction
    
    return y_pred

def print_success_rates(y_true,y_pred):
    n_success = np.sum(y_true == y_pred)   # TODO count correct prediction number
    n_total   = y_true.shape[0]    # TODO 
    print("Number of correctly labeled points: %d of %d.  Accuracy: %.2f" 
        % (n_success, n_total, n_success/n_total))

### 1.4 AI Analysis (if used)

#### Prompt(s):

#### Explanation / Changes made:

## 1.5 Predict
**Task:**

1. [1 pt] Use your custom naive Bayes to predict *TRAINING* 

In [93]:
# TODO predict training data and print
y_pred_train = naive_bayes_predict(X_train, prior, kde0, kde1)

print_success_rates(y_train, y_pred_train)

Number of correctly labeled points: 160 of 209.  Accuracy: 0.77


2. [1 pt] Use your custom naive Bayes to:
    - predict *TEST* data
    - print the results with ```print_success_rates```

In [94]:
# TODO predict test data and print
y_pred_test = naive_bayes_predict(X_test, prior, kde0, kde1)

print_success_rates(y_test, y_pred_test)

Number of correctly labeled points: 69 of 90.  Accuracy: 0.77


### 1.5 AI Analysis (if used)

#### Prompt(s):

#### Explanation / Changes made:

## 1.6 Discussion Random State
The random state changes the train test split, and could potentially change the accuracy of the model. Please answer the following questions.

**Task:**

1. [2 pts] For **custom NB**, using `random_state`=0, what is the difference between the training and test accuracy? Give an explanation for why it might be so.

The test accuracy is 0.74 while the train accuracy is 0.82. The train accuracy is slightly higher than that of the test. 
    
**Ans:** 

2. [2 pts] Now, experiment with a range of `random_state` values. Does your responses to the previous question change? If so, describe how your responses change and why you changed them.

I tried random_state=42 and obtained train 0.81 and test 0.67.
I tried random_state=600 and obtained train 0.77 and test 0.76.
I tried random_state=600000 and obtained train 0.77 and test 0.77.
I tried random_state=6000000 and obtained train 0.80 and test 0.75.


**Ans:**

### 1.6 AI Analysis (if used)

#### Prompt(s):

#### Explanation / Changes made:

##### 2 LDA and QDA

In this section you will demonstrate your understanding of LDA and QDA by completing the following functions.

## 2.1 LDA Implementation

Complete the following code block to implement the `lda_predict` function. Most of the variables have already been named you just need to assign them.

**Task:**

- [2 pts] Calculate prior of each class assuming a uniform prior
- [1 pt] Split the training data into two seperate class specific datasets
- [1 pt] Calculate the mean of each class using `np.mean`
- [2 pts] Calculate the covariance matrix for each class using `np.cov`
- [2 pt] Calculate the likelihoods of each class using `multivariate_normal.pdf()`
- [2 pt] Calculate the posterior for each class
- [1 pt] Return the predicted classifications

HINTS:
- Be careful with transposes and axis declarations 
    - Make sure to check the shapes of your calculated matrices to make sure they make sense in the context of the problem
- `np.where` might be helpful... https://numpy.org/doc/stable/reference/generated/numpy.where.html

In [110]:
def lda_predict(X_train, y_train, X_test):
    
    prior_class_0 = np.mean(y_train==0) # TODO: prior likelihood of class 0
    prior_class_1 = np.mean(y_train==1) # TODO: prior likelihood of class 1

    X_class_0 = X_train[y_train==0, :] # TODO: Seperate X_train by class
    X_class_1 = X_train[y_train==1, :] # TODO:
    
    mu_class_0 = np.mean(X_class_0, axis=0) # TODO: Mean
    mu_class_1 = np.mean(X_class_1, axis=0) # TODO: Mean

    sigma_class_0 = np.cov(X_train.T) # TODO: Proper covariance matrix for LDA
    sigma_class_1 = sigma_class_0 # TODO: Proper covariance matrix for LDA

    likelihood_class_0 = multivariate_normal.pdf(X_test, mean = mu_class_0, cov = sigma_class_0) # TODO: Calculate likelihood
    likelihood_class_1 = multivariate_normal.pdf(X_test, mean = mu_class_1, cov = sigma_class_1) # TODO: Calculate likelihood

    posterior_class_0 = (likelihood_class_0 * prior_class_0) # TODO: Posterior 
    posterior_class_1 = (likelihood_class_1 * prior_class_1) # TODO: Posterior

    y_pred = (posterior_class_1 > posterior_class_0).astype(int)
    return y_pred # TODO: return predicted class

### 2.1 AI Analysis (if used)

#### Prompt(s):

#### Explanation / Changes made:

## 2.2 QDA Implementation

Complete the following code block to implement the `qda_predict` function. Most of the variables have already been named you just need to assign them. You can repeat much of the same code from LDA. Just don't get too overzealous and forget something ;)

**Task**

- [2 pts] Calculate prior of each class assuming a uniform prior
- [1 pt] Split the training data into two seperate class specific datasets
- [1 pt] Calculate the mean of each class using `np.mean`
- [2 pts] Calculate the covariance matrix for each class using `np.cov`
- [2 pts] Calculate the likelihoods of each class `multivariate_normal.pdf()`
- [2 pts] Calculate the posterior for each class
- [1 pt] Return the predicted classifications

In [111]:
def qda_predict(X_train, y_train, X_test):
    prior_class_0 = np.mean(y_train==0) # TODO: prior likelihood of class 0
    prior_class_1 = np.mean(y_train==1) # TODO: prior likelihood of class 1

    X_class_0 = X_train[y_train==0, :] # TODO: Seperate X_train by class
    X_class_1 = X_train[y_train==1, :] # TODO:
    
    mu_class_0 = np.mean(X_class_0, axis=0) # TODO: Mean
    mu_class_1 = np.mean(X_class_1, axis=0) # TODO: Mean

    sigma_class_0 = np.cov(X_class_0.T) # TODO: Proper covariance matrix for LDA
    sigma_class_1 = np.cov(X_class_1.T) # TODO: Proper covariance matrix for LDA

    likelihood_class_0 = multivariate_normal.pdf(X_test, mean = mu_class_0, cov = sigma_class_0) # TODO: Calculate likelihood
    likelihood_class_1 = multivariate_normal.pdf(X_test, mean = mu_class_1, cov = sigma_class_1) # TODO: Calculate likelihood

    posterior_class_0 = (likelihood_class_0 * prior_class_0) # TODO: Posterior 
    posterior_class_1 = (likelihood_class_1 * prior_class_1) # TODO: Posterior

    y_pred = (posterior_class_1 > posterior_class_0).astype(int)
    return y_pred # TODO: return predicted class

### 2.2 AI Analysis (if used)

#### Prompt(s):

#### Explanation / Changes made:

## 2.3 Discussion

Run the code below to predict the test data and print the success rate, then answer the question.


In [112]:
y_pred_lda = lda_predict(X_train, y_train, X_test)
y_pred_qda = qda_predict(X_train, y_train, X_test)

print_success_rates(y_pred_lda, y_test)
print_success_rates(y_pred_qda, y_test)

Number of correctly labeled points: 69 of 90.  Accuracy: 0.77
Number of correctly labeled points: 69 of 90.  Accuracy: 0.77


**Task:**

[2 pts] Note the success rates of the two methods. Are they the same or different? If they are the same does that mean that they had the exact same predictions for each sample? Use the code block below to support your answer.

**Ans:**

LDA and QDA have the same accuracy for the test set. This does not mean they have the same predictions for each sample. See below, where an element-wise comparison is done for the predictions of LDA and QDA. 

In [124]:
# TODO: Code for 2.3

print(np.sum(y_pred_lda==y_pred_qda) == len(y_pred_qda))

False


### 2.3 AI Analysis (if used)

#### Prompt(s):

#### Explanation / Changes made:

## <span style="color:red"> Before submitting your hw, set train test split to random_state=0. Restart kernel and rerun all cells. </span>